# QS World University Rankings — Exploratory Data Analysis

This notebook loads the scraped and cleaned data from SQLite and produces key visualisations:
1. Geographic heatmap — Top 100 universities per country
2. Metric correlation matrix
3. Research vs Learning Experience by continent
4. Top 20 universities by overall score
5. Summary statistics

> **Pre-requisite:** Run `scraper.py` → `pipeline.py` before executing this notebook.

In [ ]:
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import seaborn as sns

DB_PATH = Path("data/processed/qs_rankings.db")

if not DB_PATH.exists():
    raise FileNotFoundError(
        f"Database not found at {DB_PATH}.\n"
        "Run: python scraper.py && python pipeline.py"
    )

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql("SELECT * FROM v_full_rankings ORDER BY rank", conn)
conn.close()

print(f"Loaded {len(df)} universities")
df.head()

In [ ]:
df.info()
print("\nMissing values per column:")
print(df.isnull().sum())

## 1. Geographic Heatmap — Top 100 Universities per Country

In [ ]:
top100 = df[df["rank"] <= 100].copy()
count_by_country = (
    top100.groupby("country")
    .size()
    .reset_index(name="university_count")
    .sort_values("university_count", ascending=False)
)

fig = px.choropleth(
    count_by_country,
    locations="country",
    locationmode="country names",
    color="university_count",
    color_continuous_scale="Blues",
    title="Count of Top 100 Universities by Country",
    labels={"university_count": "# Universities"},
)
fig.update_layout(geo=dict(showframe=False, showcoastlines=True))
fig.show()

print("\nTop 10 countries by university count in top 100:")
print(count_by_country.head(10).to_string(index=False))

## 2. Metric Correlation Matrix

In [ ]:
numeric_cols = [
    "overall_score", "citations_per_faculty", "academic_reputation",
    "employer_reputation", "intl_faculty_ratio", "intl_student_ratio",
    "research_discovery", "learning_experience", "employability",
    "global_engagement", "sustainability",
]
available = [c for c in numeric_cols if c in df.columns]
corr = df[available].dropna(how="all").corr()

plt.figure(figsize=(12, 10))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5,
)
plt.title("QS Metrics Correlation Matrix", fontsize=14)
plt.tight_layout()
plt.show()

## 3. Research vs Learning Experience by Continent

In [ ]:
continent_avg = (
    df.groupby("continent")[["research_discovery", "learning_experience"]]
    .mean()
    .dropna(how="all")
    .reset_index()
)

# Plotly version
fig = px.bar(
    continent_avg.melt(
        id_vars="continent",
        value_vars=["research_discovery", "learning_experience"],
        var_name="Metric",
        value_name="Average Score",
    ),
    x="continent",
    y="Average Score",
    color="Metric",
    barmode="group",
    title="Research & Discovery vs Learning Experience by Continent",
    labels={"continent": "Continent"},
    color_discrete_map={
        "research_discovery": "#636EFA",
        "learning_experience": "#EF553B",
    },
)
fig.update_layout(xaxis_tickangle=-30)
fig.show()

# Matplotlib / Seaborn version for static export
continent_avg.set_index("continent")[["research_discovery", "learning_experience"]].plot(
    kind="bar", figsize=(12, 6), color=["steelblue", "tomato"]
)
plt.title("Research & Discovery vs Learning Experience by Continent", fontsize=13)
plt.ylabel("Average Score")
plt.xlabel("Continent")
plt.xticks(rotation=30, ha="right")
plt.legend(["Research & Discovery", "Learning Experience"])
plt.tight_layout()
plt.show()

## 4. Top 20 Universities by Overall Score

In [ ]:
top20 = df[df["overall_score"].notna()].nsmallest(20, "rank")

fig = px.bar(
    top20,
    x="overall_score",
    y="university_name",
    orientation="h",
    color="continent",
    title="Top 20 Universities — Overall Score",
    labels={"overall_score": "Overall Score", "university_name": ""},
    text="overall_score",
)
fig.update_traces(texttemplate="%{text:.1f}", textposition="outside")
fig.update_layout(
    yaxis={"categoryorder": "total ascending"},
    height=600,
)
fig.show()

## 5. Summary Statistics

In [ ]:
summary = df[available].describe().T
summary.style.background_gradient(cmap="YlGn", subset=["mean", "50%"]).format("{:.2f}")

In [ ]:
# Country distribution of all scraped universities
country_dist = df["country"].value_counts().head(20).reset_index()
country_dist.columns = ["country", "count"]

fig = px.bar(
    country_dist,
    x="count",
    y="country",
    orientation="h",
    title="Top 20 Countries by Number of Ranked Universities",
    labels={"count": "Number of Universities", "country": ""},
    color="count",
    color_continuous_scale="Teal",
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()